# 00 · Setup and connect

Install the SDK, authenticate, and read the **event clock** — so you know which phase is
running and what you should be doing right now. Two minutes, no model training.

The event runs as a **~3-hour build-and-validate phase** followed by **four sealed rounds** of
~30 minutes each — one event's configuration, not a rule, so read the real clock off the cells
below rather than planning against those numbers. Only the round currently open is being
scored, and per-round scores accumulate into the cumulative standings that decide a
display-only event (a money event is decided by the final stake balance instead).
Full description in the [README](../README.md#the-event).

**Next:** [`01_explore_the_data.ipynb`](01_explore_the_data.ipynb) →
[`02_train_and_submit.ipynb`](02_train_and_submit.ipynb)

## 1. Install

In [ ]:
# everestapi is the Everesteer SDK. This notebook only needs the SDK and a parquet reader.
%pip install --quiet "everestapi>=0.3.32" pandas pyarrow

## 2. Authenticate

Credentials come from onboarding's **Copy setup command** (*Install & connect your agent → Step 2*).
Set them in your shell **before** launching Jupyter (or in a local `.env`) — never commit them:

`EIQ_API_KEY`, `EIQ_BASE_URL` (defaults to `https://app.everesteer.ai`)

`CF_ACCESS_CLIENT_ID` / `CF_ACCESS_CLIENT_SECRET` are only needed against the gated
**staging** environment — omit them on the public site.

In [ ]:
import os
from everestapi import EverestAPI

# Read every credential from the environment, so nothing secret is written into the notebook.
base_url = os.environ.get("EIQ_BASE_URL", "https://app.everesteer.ai")
api_key = os.environ.get("EIQ_API_KEY") or os.environ.get("EVEREST_API_KEY")

# Fail fast with a clear message instead of a cryptic 401/403 deeper in the notebook.
if not api_key:
    raise RuntimeError("Set EIQ_API_KEY (onboarding -> Copy setup command).")

# Only the gated STAGING mirror needs a Cloudflare Access service token; the public
# site does not. The SDK picks the CF_ACCESS_* env vars up automatically when set.
if "staging" in base_url and not (
    os.environ.get("CF_ACCESS_CLIENT_ID") and os.environ.get("CF_ACCESS_CLIENT_SECRET")
):
    raise RuntimeError(
        "Staging is behind Cloudflare Access - set CF_ACCESS_CLIENT_ID / "
        "CF_ACCESS_CLIENT_SECRET, or point EIQ_BASE_URL at the public site."
    )

client = EverestAPI(api_key=api_key, base_url=base_url)
client.health()  # returns {'status': 'ok', ...} on a working, authenticated connection

## 3. Where are we in the event?

`get_started` is mode-aware: it reports the shape of *your* event, including a `cadence`
object when the event runs on a clock. **Read the phase, never assume it** — round lengths
are configured per event and can be paused or extended on the day.

The cell below turns that object into a sentence.

In [ ]:
started = client.get_started() or {}
cadence = started.get("cadence") or {}
# get_started's event_staking block is the ONLY authority on whether this event
# carries money (do not infer it from anywhere else) - see AGENTS.md#event-staking.
event_staking = started.get("event_staking") or {}

print("scope:", started.get("scope") or started.get("mode") or "unknown")

if not cadence:
    print("\nNo cadence on this key - either a legacy no-clock event, or a tournament key.")
else:
    phase = cadence.get("phase")
    open_window = cadence.get("open_window")
    fenced = cadence.get("intake_fenced")
    ends_at = cadence.get("phase_ends_at")
    secs = cadence.get("seconds_until_next_phase")

    # Say what is happening, and what it means for you.
    if phase == "build":
        headline = ("BUILD & VALIDATE - fit on the labeled `train` split, and check yourself on "
                    "the practice board. Nothing here counts toward the standings.")
    elif open_window:
        headline = f"ROUND OPEN ({open_window}) - predict the `live` split and submit."
    elif phase in ("done", "complete"):
        if event_staking:
            headline = ("EVENT COMPLETE - this was a staked event: the final recorded "
                        "STAKE BALANCE decides the winner, not the standings table.")
        else:
            headline = "EVENT COMPLETE - the cumulative standings are final."
    else:
        headline = f"phase: {phase}"
    print("\n" + headline)

    if fenced:
        print("  ! intake is FENCED right now (a round is settling) - uploads are refused; wait it out.")
    if ends_at:
        print(f"  phase ends at: {ends_at}")
    if secs is not None:
        print(f"  next phase in: {int(secs) // 60} min {int(secs) % 60} s")

    plan = cadence.get("round_minutes_plan") or cadence.get("phase_sequence")
    if plan:
        print(f"  schedule: {plan}")

## 4. The dataset schema

The schema is the map of the data: which feature sets exist, how big each one is, and which
targets you can train on.

Two things the schema will not tell you if you guess instead of read it:

- **Feature-set membership is not derivable from a feature's name.** The middle word of
  `feature_<adj>_<noun>_<adj>` is drawn from a fixed vocabulary and carries no grouping
  information — and membership is many-to-many, which no single token could encode.
- **`-1` in a feature value means missing**, not "a bin below zero".

In [ ]:
schema = client.get_dataset_schema(verbose=True)
feature_sets = schema["feature_sets"]
target_names = list(schema["targets"])

# Which feature sets exist is a dataset fact: some publish size tiers
# (small/medium/all), some publish only one set. Read them, never assume.
for s in sorted(feature_sets, key=lambda n: len(feature_sets[n])):
    print(f"feature set {s!r:>9}: {len(feature_sets[s]):>4} features")

# The graded column, straight from the schema. It differs between datasets and
# is NOT necessarily the first entry in `targets` - never hardcode it.
PRIMARY_TARGET = schema["primary_target"]
print(f"\ntargets: {len(target_names)} -> {target_names[:4]} ...")
print("scored target:", PRIMARY_TARGET)

## 5. What you have to spend

Two budgets worth knowing before the first round opens:

- **Uploads** are capped **per account, not per round** — every round draws from the same
  allowance.
- **Compute credits** fund hosted training (`client.train(...)`). In an event the grant is
  usually already spendable; a CPU LightGBM baseline costs well under $1.

In [ ]:
status = {}
try:
    status = client.get_status() or {}
    print("uploads remaining:", status.get("uploads_remaining", "not reported"))
except Exception as e:
    print("get_status unavailable:", e)

try:
    credits = client.get_compute_credits() or {}
    print("compute credits:", credits.get("balance_usd", credits))
except Exception as e:
    print("compute credits unavailable:", e)

print("\nhosted train funded:", started.get("hosted_train_funded", "not reported"))
print("suggested next actions:", started.get("next_actions", []))

## You're connected

- Green `health()`, a phase printed above, and a schema you can read → you're set up.
- **Now:** [`01_explore_the_data.ipynb`](01_explore_the_data.ipynb) to see what you're modelling.
- **Then:** [`02_train_and_submit.ipynb`](02_train_and_submit.ipynb) to fit a baseline and enter a round.

Agents driving the tools directly should read [`AGENTS.md`](../AGENTS.md) instead — it carries
the full loop, the staking surface, and the research skills.